In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from google.colab import files
from scipy.ndimage import uniform_filter

uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def triangular_fuzzy_filter_fast(image, kernel_size=3):
    pad = kernel_size // 2

    xmav = uniform_filter(image.astype(np.float32), size=(kernel_size, kernel_size, 1))

    min_filter = np.zeros_like(image, dtype=np.float32)
    max_filter = np.zeros_like(image, dtype=np.float32)

    for ch in range(image.shape[2]):
        min_filter[:, :, ch] = cv2.erode(image[:, :, ch], np.ones((kernel_size, kernel_size)))
        max_filter[:, :, ch] = cv2.dilate(image[:, :, ch], np.ones((kernel_size, kernel_size)))

    xmv = np.maximum(max_filter - xmav, xmav - min_filter)
    xmv[xmv == 0] = 1e-6

    fuzzy = 1 - np.abs(image - xmav) / xmv
    fuzzy = np.clip(fuzzy, 0, 1)

    num = uniform_filter((fuzzy * image).astype(np.float32), size=(kernel_size, kernel_size, 1))
    den = uniform_filter(fuzzy.astype(np.float32), size=(kernel_size, kernel_size, 1))

    filtered = num / (den + 1e-6)
    return np.clip(filtered, 0, 255).astype(np.uint8)


def unsharp_mask(img, amount=1.7, blur=3):
    blurred = cv2.GaussianBlur(img, (blur, blur), 0)
    return cv2.addWeighted(img, 1 + amount, blurred, -amount, 0)


def enhance_brightness_contrast(img, brightness=8, contrast=1.18):
    return cv2.convertScaleAbs(img, alpha=contrast, beta=brightness)


def detailed_sharpen(img):
    kernel = np.array([[0, -1, 0],
                       [-1, 7.5, -1],
                       [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)


fuzzy_filtered = triangular_fuzzy_filter_fast(img_rgb, kernel_size=3)

sharp1 = unsharp_mask(fuzzy_filtered, amount=1.7, blur=3)

enhanced_bc = enhance_brightness_contrast(sharp1, brightness=8, contrast=1.18)

final_output = detailed_sharpen(enhanced_bc)


plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(final_output)
plt.title("Enhanced Image")
plt.axis("off")
plt.tight_layout()
plt.show()


output_path = "enhanced_image_sharp.png"
Image.fromarray(final_output).save(output_path)
files.download(output_path)


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from google.colab import files
from PIL import Image

# -------------------- UPLOAD --------------------
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# -------------------- 1. LIGHT PROTECTION --------------------
def protect_highlights(img):
    img = img.astype(np.float32) / 255.0
    img = img ** 0.85      # compress highlights safely
    return np.clip(img * 255, 0, 255).astype(np.uint8)


# -------------------- 2. NIGHT BRIGHTENING --------------------
def brighten_night(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    # Adaptive brightening
    l = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)

    enhanced = cv2.merge((l, a, b))
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)


# -------------------- 3. NOISE REMOVAL --------------------
def denoise(img):
    return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)


# -------------------- 4. SOFT SHARPENING (SAFE) --------------------
def soft_sharpen(img):
    blur = cv2.GaussianBlur(img, (3,3), 0)
    return cv2.addWeighted(img, 1.15, blur, -0.15, 0)


# -------------------- PIPELINE --------------------
step1 = protect_highlights(img)
step2 = brighten_night(step1)
step3 = denoise(step2)
final_output = soft_sharpen(step3)

# -------------------- DISPLAY --------------------
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(final_output)
plt.title("Night Enhanced (Clean & Natural)")
plt.axis("off")

plt.show()

# -------------------- SAVE --------------------
output_path = "night_enhanced.png"
Image.fromarray(final_output).save(output_path)
files.download(output_path)


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from google.colab import files
from PIL import Image

# -------------------- UPLOAD --------------------
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# -------------------- 1. LIGHT PROTECTION --------------------
def protect_highlights(img):
    img = img.astype(np.float32) / 255.0
    img = img ** 0.85      # compress highlights safely
    return np.clip(img * 255, 0, 255).astype(np.uint8)


# -------------------- 2. NIGHT BRIGHTENING --------------------
def brighten_night(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    # Adaptive brightening
    l = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)

    enhanced = cv2.merge((l, a, b))
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)


# -------------------- 3. NOISE REMOVAL --------------------
def denoise(img):
    return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)


# -------------------- 4. SOFT SHARPENING (SAFE) --------------------
def soft_sharpen(img):
    blur = cv2.GaussianBlur(img, (3,3), 0)
    return cv2.addWeighted(img, 1.15, blur, -0.15, 0)


# -------------------- PIPELINE --------------------
step1 = protect_highlights(img)
step2 = brighten_night(step1)
step3 = denoise(step2)
final_output = soft_sharpen(step3)

# -------------------- DISPLAY --------------------
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(final_output)
plt.title("Night Enhanced (Clean & Natural)")
plt.axis("off")

plt.show()

# -------------------- SAVE --------------------
output_path = "night_enhanced.png"
Image.fromarray(final_output).save(output_path)
files.download(output_path)


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from google.colab import files
from scipy.ndimage import uniform_filter

uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def triangular_fuzzy_filter_fast(image, kernel_size=3):
    pad = kernel_size // 2

    xmav = uniform_filter(image.astype(np.float32), size=(kernel_size, kernel_size, 1))

    min_filter = np.zeros_like(image, dtype=np.float32)
    max_filter = np.zeros_like(image, dtype=np.float32)

    for ch in range(image.shape[2]):
        min_filter[:, :, ch] = cv2.erode(image[:, :, ch], np.ones((kernel_size, kernel_size)))
        max_filter[:, :, ch] = cv2.dilate(image[:, :, ch], np.ones((kernel_size, kernel_size)))

    xmv = np.maximum(max_filter - xmav, xmav - min_filter)
    xmv[xmv == 0] = 1e-6

    fuzzy = 1 - np.abs(image - xmav) / xmv
    fuzzy = np.clip(fuzzy, 0, 1)

    num = uniform_filter((fuzzy * image).astype(np.float32), size=(kernel_size, kernel_size, 1))
    den = uniform_filter(fuzzy.astype(np.float32), size=(kernel_size, kernel_size, 1))

    filtered = num / (den + 1e-6)
    return np.clip(filtered, 0, 255).astype(np.uint8)


def unsharp_mask(img, amount=1.7, blur=3):
    blurred = cv2.GaussianBlur(img, (blur, blur), 0)
    return cv2.addWeighted(img, 1 + amount, blurred, -amount, 0)


def enhance_brightness_contrast(img, brightness=8, contrast=1.18):
    return cv2.convertScaleAbs(img, alpha=contrast, beta=brightness)


def detailed_sharpen(img):
    kernel = np.array([[0, -1, 0],
                       [-1, 7.5, -1],
                       [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)


fuzzy_filtered = triangular_fuzzy_filter_fast(img_rgb, kernel_size=3)

sharp1 = unsharp_mask(fuzzy_filtered, amount=1.7, blur=3)

enhanced_bc = enhance_brightness_contrast(sharp1, brightness=8, contrast=1.18)

final_output = detailed_sharpen(enhanced_bc)


plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(final_output)
plt.title("Enhanced + Sharpened Image")
plt.axis("off")
plt.tight_layout()
plt.show()


output_path = "enhanced_image_sharp.png"
Image.fromarray(final_output).save(output_path)
files.download(output_path)


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from google.colab import files
from scipy.ndimage import uniform_filter

uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def triangular_fuzzy_filter_fast(image, kernel_size=3):
    pad = kernel_size // 2

    xmav = uniform_filter(image.astype(np.float32), size=(kernel_size, kernel_size, 1))

    min_filter = np.zeros_like(image, dtype=np.float32)
    max_filter = np.zeros_like(image, dtype=np.float32)

    for ch in range(image.shape[2]):
        min_filter[:, :, ch] = cv2.erode(image[:, :, ch], np.ones((kernel_size, kernel_size)))
        max_filter[:, :, ch] = cv2.dilate(image[:, :, ch], np.ones((kernel_size, kernel_size)))

    xmv = np.maximum(max_filter - xmav, xmav - min_filter)
    xmv[xmv == 0] = 1e-6


    fuzzy = 1 - np.abs(image - xmav) / xmv
    fuzzy = np.clip(fuzzy, 0, 1)


    num = uniform_filter((fuzzy * image).astype(np.float32), size=(kernel_size, kernel_size, 1))
    den = uniform_filter(fuzzy.astype(np.float32), size=(kernel_size, kernel_size, 1))

    filtered = num / (den + 1e-6)
    return np.clip(filtered, 0, 255).astype(np.uint8)

def unsharp_mask(img, amount=1.7, blur=3):
    blurred = cv2.GaussianBlur(img, (blur, blur), 0)
    return cv2.addWeighted(img, 1 + amount, blurred, -amount, 0)


def enhance_brightness_contrast(img, brightness=8, contrast=1.18):
    return cv2.convertScaleAbs(img, alpha=contrast, beta=brightness)


def detailed_sharpen(img):
    kernel = np.array([[0, -1, 0],
                       [-1, 7.5, -1],
                       [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)



fuzzy_filtered = triangular_fuzzy_filter_fast(img_rgb, kernel_size=3)

sharp1 = unsharp_mask(fuzzy_filtered, amount=1.7, blur=3)

enhanced_bc = enhance_brightness_contrast(sharp1, brightness=8, contrast=1.18)

final_output = detailed_sharpen(enhanced_bc)



plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(final_output)
plt.title("Enhanced + Sharpened Image")
plt.axis("off")
plt.tight_layout()
plt.show()


output_path = "enhanced_image_sharp.png"
Image.fromarray(final_output).save(output_path)
files.download(output_path)


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from google.colab import files
from scipy.ndimage import uniform_filter

uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def triangular_fuzzy_filter_fast(image, kernel_size=3):
    pad = kernel_size // 2

    xmav = uniform_filter(image.astype(np.float32), size=(kernel_size, kernel_size, 1))

    min_filter = np.zeros_like(image, dtype=np.float32)
    max_filter = np.zeros_like(image, dtype=np.float32)

    for ch in range(image.shape[2]):
        min_filter[:, :, ch] = cv2.erode(image[:, :, ch], np.ones((kernel_size, kernel_size)))
        max_filter[:, :, ch] = cv2.dilate(image[:, :, ch], np.ones((kernel_size, kernel_size)))

    xmv = np.maximum(max_filter - xmav, xmav - min_filter)
    xmv[xmv == 0] = 1e-6

    fuzzy = 1 - np.abs(image - xmav) / xmv
    fuzzy = np.clip(fuzzy, 0, 1)

    num = uniform_filter((fuzzy * image).astype(np.float32), size=(kernel_size, kernel_size, 1))
    den = uniform_filter(fuzzy.astype(np.float32), size=(kernel_size, kernel_size, 1))

    filtered = num / (den + 1e-6)
    return np.clip(filtered, 0, 255).astype(np.uint8)


def unsharp_mask(img, amount=1.7, blur=3):
    blurred = cv2.GaussianBlur(img, (blur, blur), 0)
    return cv2.addWeighted(img, 1 + amount, blurred, -amount, 0)


def enhance_brightness_contrast(img, brightness=8, contrast=1.18):
    return cv2.convertScaleAbs(img, alpha=contrast, beta=brightness)


def detailed_sharpen(img):
    kernel = np.array([[0, -1, 0],
                       [-1, 7.5, -1],
                       [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)


fuzzy_filtered = triangular_fuzzy_filter_fast(img_rgb, kernel_size=3)

sharp1 = unsharp_mask(fuzzy_filtered, amount=1.7, blur=3)

enhanced_bc = enhance_brightness_contrast(sharp1, brightness=8, contrast=1.18)

final_output = detailed_sharpen(enhanced_bc)


plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(final_output)
plt.title("Enhanced + Sharpened Image")
plt.axis("off")
plt.tight_layout()
plt.show()


output_path = "enhanced_image_sharp.png"
Image.fromarray(final_output).save(output_path)
files.download(output_path)


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from google.colab import files
from scipy.ndimage import uniform_filter

uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def triangular_fuzzy_filter_fast(image, kernel_size=3):
    pad = kernel_size // 2

    xmav = uniform_filter(image.astype(np.float32), size=(kernel_size, kernel_size, 1))

    min_filter = np.zeros_like(image, dtype=np.float32)
    max_filter = np.zeros_like(image, dtype=np.float32)

    for ch in range(image.shape[2]):
        min_filter[:, :, ch] = cv2.erode(image[:, :, ch], np.ones((kernel_size, kernel_size)))
        max_filter[:, :, ch] = cv2.dilate(image[:, :, ch], np.ones((kernel_size, kernel_size)))

    xmv = np.maximum(max_filter - xmav, xmav - min_filter)
    xmv[xmv == 0] = 1e-6

    fuzzy = 1 - np.abs(image - xmav) / xmv
    fuzzy = np.clip(fuzzy, 0, 1)

    num = uniform_filter((fuzzy * image).astype(np.float32), size=(kernel_size, kernel_size, 1))
    den = uniform_filter(fuzzy.astype(np.float32), size=(kernel_size, kernel_size, 1))

    filtered = num / (den + 1e-6)
    return np.clip(filtered, 0, 255).astype(np.uint8)


def unsharp_mask(img, amount=1.7, blur=3):
    blurred = cv2.GaussianBlur(img, (blur, blur), 0)
    return cv2.addWeighted(img, 1 + amount, blurred, -amount, 0)


def enhance_brightness_contrast(img, brightness=8, contrast=1.18):
    return cv2.convertScaleAbs(img, alpha=contrast, beta=brightness)


def detailed_sharpen(img):
    kernel = np.array([[0, -1, 0],
                       [-1, 7.5, -1],
                       [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)


fuzzy_filtered = triangular_fuzzy_filter_fast(img_rgb, kernel_size=3)

sharp1 = unsharp_mask(fuzzy_filtered, amount=1.7, blur=3)

enhanced_bc = enhance_brightness_contrast(sharp1, brightness=8, contrast=1.18)

final_output = detailed_sharpen(enhanced_bc)


plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(final_output)
plt.title("Enhanced + Sharpened Image")
plt.axis("off")
plt.tight_layout()
plt.show()


output_path = "enhanced_image_sharp.png"
Image.fromarray(final_output).save(output_path)
files.download(output_path)


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from google.colab import files
from PIL import Image

# -------------------- UPLOAD --------------------
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# -------------------- 1. LIGHT PROTECTION --------------------
def protect_highlights(img):
    img = img.astype(np.float32) / 255.0
    img = img ** 0.85      # compress highlights safely
    return np.clip(img * 255, 0, 255).astype(np.uint8)


# -------------------- 2. NIGHT BRIGHTENING --------------------
def brighten_night(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    # Adaptive brightening
    l = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)

    enhanced = cv2.merge((l, a, b))
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)


# -------------------- 3. NOISE REMOVAL --------------------
def denoise(img):
    return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)


# -------------------- 4. SOFT SHARPENING (SAFE) --------------------
def soft_sharpen(img):
    blur = cv2.GaussianBlur(img, (3,3), 0)
    return cv2.addWeighted(img, 1.15, blur, -0.15, 0)


# -------------------- PIPELINE --------------------
step1 = protect_highlights(img)
step2 = brighten_night(step1)
step3 = denoise(step2)
final_output = soft_sharpen(step3)

# -------------------- DISPLAY --------------------
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(final_output)
plt.title("Night Enhanced (Clean & Natural)")
plt.axis("off")

plt.show()

# -------------------- SAVE --------------------
output_path = "night_enhanced.png"
Image.fromarray(final_output).save(output_path)
files.download(output_path)


In [ ]:
# -----------------------------------------
# 1. Upload image from your laptop
# -----------------------------------------
from google.colab import files
uploaded = files.upload()   # choose your image file

# -----------------------------------------
# 2. Read the uploaded image using OpenCV
# -----------------------------------------
import cv2
import numpy as np

image_path = list(uploaded.keys())[0]   # gets the filename automatically
img = cv2.imread(image_path)

if img is None:
    raise ValueError("Image failed to load. Try uploading again.")

print("Image Loaded Successfully:", image_path)

# -----------------------------------------
# 3. FAST vectorized entropy function
# -----------------------------------------
def calc_entropy(gray_img):
    # Histogram
    hist = cv2.calcHist([gray_img], [0], None, [256], [0,256]).ravel()
    p = hist / hist.sum()   # normalize to probability
    p = p[p > 0]            # remove zeros
    return -np.sum(p * np.log2(p))


# -----------------------------------------
# 4. Entropy for Grayscale
# -----------------------------------------
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
entropy_gray = calc_entropy(gray)
print("Grayscale Entropy:", entropy_gray)

# -----------------------------------------
# 5. Entropy for RGB Channels Separately
# -----------------------------------------
# OpenCV loads channels as BGR → convert to RGB
b, g, r = cv2.split(img)

entropy_r = calc_entropy(r)
entropy_g = calc_entropy(g)
entropy_b = calc_entropy(b)

print("Entropy - Red Channel:   ", entropy_r)
print("Entropy - Green Channel: ", entropy_g)
print("Entropy - Blue Channel:  ", entropy_b)

# -----------------------------------------
# 6. Optional: Overall RGB average entropy
# -----------------------------------------
entropy_rgb_avg = (entropy_r + entropy_g + entropy_b) / 3
print("Average RGB Entropy:", entropy_rgb_avg)


In [ ]:
# ----------------------------------------------------------
# 1. Upload image from your laptop
# ----------------------------------------------------------
from google.colab import files
uploaded = files.upload()   # Choose any image from your laptop

# ----------------------------------------------------------
# 2. Import Libraries
# ----------------------------------------------------------
import cv2
import numpy as np

# ----------------------------------------------------------
# 3. Take uploaded image name automatically
# ----------------------------------------------------------
image_path = list(uploaded.keys())[0]
img = cv2.imread(image_path)

if img is None:
    raise ValueError("Error: Image failed to load. Upload again.")

print("Image Loaded Successfully:", image_path)

# ----------------------------------------------------------
# 4. Shannon Entropy Function (FAST + Vectorized)
# ----------------------------------------------------------
def shannon_entropy(gray_img):
    # Histogram 0–255
    hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256]).ravel()

    # Probability distribution
    p = hist / hist.sum()

    # Remove zero probabilities
    p = p[p > 0]

    # Shannon formula
    return -np.sum(p * np.log2(p))

# ----------------------------------------------------------
# 5. Grayscale Entropy
# ----------------------------------------------------------
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
entropy_gray = shannon_entropy(gray)
print("Grayscale Entropy:", entropy_gray)

# ----------------------------------------------------------
# 6. RGB Channel Entropy
# ----------------------------------------------------------
b, g, r = cv2.split(img)

entropy_r = shannon_entropy(r)
entropy_g = shannon_entropy(g)
entropy_b = shannon_entropy(b)

print("Red Channel Entropy:   ", entropy_r)
print("Green Channel Entropy: ", entropy_g)
print("Blue Channel Entropy:  ", entropy_b)

# ----------------------------------------------------------
# 7. Optional: Average RGB Entropy
# ----------------------------------------------------------
entropy_avg = (entropy_r + entropy_g + entropy_b) / 3
print("Average RGB Entropy:", entropy_avg)


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from google.colab import files
from scipy.ndimage import uniform_filter

uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def triangular_fuzzy_filter_fast(image, kernel_size=3):
    pad = kernel_size // 2
    xmav = uniform_filter(image.astype(np.float32), size=(kernel_size, kernel_size, 1))

    min_filter = np.zeros_like(image, dtype=np.float32)
    max_filter = np.zeros_like(image, dtype=np.float32)

    for ch in range(image.shape[2]):
        min_filter[:, :, ch] = cv2.erode(image[:, :, ch], np.ones((kernel_size, kernel_size)))
        max_filter[:, :, ch] = cv2.dilate(image[:, :, ch], np.ones((kernel_size, kernel_size)))

    xmv = np.maximum(max_filter - xmav, xmav - min_filter)
    xmv[xmv == 0] = 1e-6

    fuzzy = 1 - np.abs(image - xmav) / xmv
    fuzzy = np.clip(fuzzy, 0, 1)

    num = uniform_filter((fuzzy * image).astype(np.float32), size=(kernel_size, kernel_size, 1))
    den = uniform_filter(fuzzy.astype(np.float32), size=(kernel_size, kernel_size, 1))

    filtered = num / (den + 1e-6)
    return np.clip(filtered, 0, 255).astype(np.uint8)


def unsharp_mask(img, amount=1.7, blur=3):
    blurred = cv2.GaussianBlur(img, (blur, blur), 0)
    return cv2.addWeighted(img, 1 + amount, blurred, -amount, 0)


def enhance_brightness_contrast(img, brightness=8, contrast=1.18):
    return cv2.convertScaleAbs(img, alpha=contrast, beta=brightness)


def detailed_sharpen(img):
    kernel = np.array([[0, -1, 0],
                       [-1, 7.5, -1],
                       [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)


gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

highlight_mask = (gray > 210).astype(np.float32)
highlight_mask = cv2.GaussianBlur(highlight_mask, (15, 15), 0)
highlight_mask_3c = np.dstack([highlight_mask] * 3)


fuzzy_filtered = triangular_fuzzy_filter_fast(img_rgb, kernel_size=3)


sharp1 = unsharp_mask(fuzzy_filtered, amount=1.7, blur=3)

sharp1_adaptive = (sharp1 * (1 - highlight_mask_3c) +
                   fuzzy_filtered * highlight_mask_3c).astype(np.uint8)


enh_bc = enhance_brightness_contrast(sharp1_adaptive, brightness=8, contrast=1.18)

enh_bc_adaptive = (enh_bc * (1 - highlight_mask_3c) +
                   sharp1_adaptive * highlight_mask_3c).astype(np.uint8)


detail = detailed_sharpen(enh_bc_adaptive)

final_output = (detail * (1 - highlight_mask_3c) +
                enh_bc_adaptive * highlight_mask_3c).astype(np.uint8)


plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(final_output)
plt.title("Final Enhanced Image (Highlight Protected)")
plt.axis("off")
plt.tight_layout()
plt.show()

output_path = "enhanced_image_highlight_protected.png"
Image.fromarray(final_output).save(output_path)
files.download(output_path)


In [ ]:
import cv2
import numpy as np
from google.colab import files
import matplotlib.pyplot as plt

# Upload an image in Colab
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# Read the image
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Gamma correction function
def apply_gamma(image, gamma):
    inv_gamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** inv_gamma * 255
                      for i in np.arange(256)]).astype("uint8")
    return cv2.LUT(image, table)

# Different gamma values to test
gammas = [0.5, 1.0, 2.0]

plt.figure(figsize=(15,5))

for i, g in enumerate(gammas):
    corrected = apply_gamma(img, g)
    plt.subplot(1, 3, i+1)
    plt.imshow(corrected)
    plt.title(f"Gamma = {g}")
    plt.axis('off')

plt.show()
